In [8]:
import json
import os
import numpy as np
import pandas as pd

In [156]:
import floodlight.io.kinexon as knx

import floodlight.io.dfl as dfl

from floodlight.models.kinematics import DistanceModel, VelocityModel
from floodlight.models.kinetics import MetabolicPowerModel
from floodlight.models.geometry import CentroidModel


In [157]:
from floodlight.models.space import DiscreteVoronoiModel


ModuleNotFoundError: No module named 'floodlight.models.space'

## Load DFL Data

In [8]:
positions_filepath = "/home/max/drive/projects/1_SportVid/data/DFL_04_03_positions_raw_observed_DFL-COM-000001_DFL-MAT-J03WMX.xml" 
metadata_filepath = "/home/max/drive/projects/1_SportVid/data/DFL_02_01_matchinformation_DFL-COM-000001_DFL-MAT-J03WMX.xml"                                  

In [ ]:
xy, possession, ballstatus, teamsheet, pitch = dfl.read_position_data_xml(positions_filepath, metadata_filepath)                 
                                                                                                          
for key in xy:                                                                                        
    print(key)  # 'firstHalf', 'secondHalf'                                                             
    for team_name, xy_obj in xy[key].items():                                                         
      print(f'{team_name}: {xy_obj.xy.shape}')  # 'Home', 'Away', 'Ball'                              

sampled_data = xy['firstHalf']['Home'].xy                                                             
print(sampled_data)   

firstHalf
Home: (70708, 40)
Away: (70708, 40)
Ball: (70708, 2)
secondHalf
Home: (75259, 40)
Away: (75259, 40)
Ball: (75259, 2)
[[ 6.900e+00  5.120e+00        nan ...  5.180e+00        nan        nan]
 [ 6.820e+00  5.100e+00        nan ...  5.230e+00        nan        nan]
 [ 6.720e+00  5.080e+00        nan ...  5.280e+00        nan        nan]
 ...
 [-1.000e-02 -1.139e+01        nan ... -1.003e+01        nan        nan]
 [ 0.000e+00 -1.147e+01        nan ... -1.010e+01        nan        nan]
 [ 0.000e+00 -1.154e+01        nan ... -1.018e+01        nan        nan]]


In [58]:
DistMod = DistanceModel()
VelMod = VelocityModel()
MetPowMod = MetabolicPowerModel()
CentMod = CentroidModel()

kpi_dict = {}
for half in ["firstHalf", "secondHalf"]:
    for team in ["Home", "Away"]:
        print(f"{half} - {team}")
        DistMod.fit(xy[half][team])
        VelMod.fit(xy[half][team])
        MetPowMod.fit(xy[half][team])
        CentMod.fit(xy[half][team])

        kpi_dict[f'distance_covered_{half}_{team}'] = np.array(DistMod.cumulative_distance_covered())
        kpi_dict[f'max_velocity_{half}_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)
        kpi_dict[f'metabolic_power_{half}_{team}'] = np.array(MetPowMod.metabolic_power())
        kpi_dict[f'centroid_{half}_{team}'] = CentMod.centroid().xy


firstHalf - Home


/tmp/ipykernel_270336/2027585810.py:16: RuntimeWarning: All-NaN axis encountered
  kpi_dict[f'max_velocity_{half}_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)


firstHalf - Away


/tmp/ipykernel_270336/2027585810.py:16: RuntimeWarning: All-NaN axis encountered
  kpi_dict[f'max_velocity_{half}_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)


secondHalf - Home


/tmp/ipykernel_270336/2027585810.py:16: RuntimeWarning: All-NaN axis encountered
  kpi_dict[f'max_velocity_{half}_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)


secondHalf - Away


/tmp/ipykernel_270336/2027585810.py:16: RuntimeWarning: All-NaN axis encountered
  kpi_dict[f'max_velocity_{half}_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)


## Load KNX Data

In [1]:
path_knx = "/home/max/drive/projects/1_SportVid/AP1_DataGeneration/trackingDataSampleVideo.csv"

In [11]:
knx_teamsheets = knx.read_teamsheets_from_csv(path_knx)
knx_pos = knx.read_position_data_csv(path_knx)

In [26]:
knx_pos[0]

XY(xy=array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], shape=(20898, 22)), framerate=25, direction=None)

In [30]:
DistMod = DistanceModel()
VelMod = VelocityModel()
MetPowMod = MetabolicPowerModel()
CentMod = CentroidModel()

kpi_dict = {}
for team, pos in zip(["team1", "team2"], knx_pos):
    DistMod.fit(pos)
    VelMod.fit(pos)
    MetPowMod.fit(pos)
    CentMod.fit(pos)

    kpi_dict[f'distance_covered_{team}'] = np.array(DistMod.cumulative_distance_covered())
    kpi_dict[f'max_velocity_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)
    kpi_dict[f'metabolic_power_{team}'] = np.array(MetPowMod.metabolic_power())
    kpi_dict[f'centroid_{team}'] = CentMod.centroid().xy


/tmp/ipykernel_514211/1982305794.py:14: RuntimeWarning: All-NaN axis encountered
  kpi_dict[f'max_velocity_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)
/tmp/ipykernel_514211/1982305794.py:14: RuntimeWarning: All-NaN axis encountered
  kpi_dict[f'max_velocity_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)


In [33]:
kpi_dict['max_velocity_team1']

array([7.65, 8.16,  nan, 7.59, 8.87, 7.2 ,  nan,  nan, 7.82,  nan, 3.46])

In [23]:
knx_teamsheets[0].teamsheet

,player,sensor_id,mapped_id,name,number,tID,xID
0,B Kumarli,39128,373,B Kumarli,16,Group A,0
1,B Wohlan,39176,370,B Wohlan,26,Group A,1
2,B Steinhausen,39116,364,B Steinhausen,14,Group A,2
3,B Lasic,39104,367,B Lasic,6,Group A,3
4,B Reichelt,39198,371,B Reichelt,31,Group A,4
5,B Koch,39180,368,B Koch,29,Group A,5
6,B Haeck,39091,363,B Haeck,1,Group A,6
7,B Reitmeier,39160,365,B Reitmeier,23,Group A,7
8,B Vosen,39108,372,B Vosen,10,Group A,8
9,B Dolenga,39155,366,B Dolenga,18,Group A,9


## Unified KPI Data Format

Compute per-frame, per-player KPIs for both DFL and Kinexon formats.

Output structure:
```
{
  frame_idx: [
    [player_id, distance_covered, velocity, metabolic_power],
    ...  # one entry per tracked player
  ],
  ...
}
```

`meta_data` contains:
- `format`: "dfl" or "kinexon"
- `kpi_names`: ["distance_covered", "velocity", "metabolic_power"]
- `player_ids`: {combined_idx → player identifier string}
- `teams`: {team_name → [combined_idx, ...]}
- `segments`: [{"name", "start_frame", "n_frames"}, ...]  (one per half for DFL)
- `framerate`: recording framerate

In [108]:
#positions_filepath = "/home/max/drive/projects/1_SportVid/data/DFL_04_03_positions_raw_observed_DFL-COM-000001_DFL-MAT-J03WMX.xml"
positions_filepath = "/home/max/drive/projects/1_SportVid/AP1_DataGeneration/trackingDataSampleVideo.csv"
metadata_filepath = "/home/max/drive/projects/1_SportVid/data/DFL_02_01_matchinformation_DFL-COM-000001_DFL-MAT-J03WMX.xml"        

fmt = "kinexon"  # or "dfl"

## Code from Current `kpi_computation` Plugin

In [115]:
parameters = {
    "positions_filepath": positions_filepath,
    "metadata_filepath": metadata_filepath,
    "format_type": fmt,
    "delimiter": ","
}


### Parsing

In [132]:
if fmt == "kinexon":
    pos_data = knx.read_position_data_csv(positions_filepath, delimiter=parameters.get("delimiter", ";")) # pos_data is List[XY]
    teamsheets = knx.read_teamsheets_from_csv(positions_filepath) # teamsheets is List[Teamsheet]
    
    for i, (pos_xy, ts) in enumerate(zip(pos_data, teamsheets)):
        team_name = ts.teamsheet['tID'].iloc[0] if not ts.teamsheet.empty else f"team_{i+1}"
        df_ts = ts.teamsheet.sort_values("xID")
        team_players[team_name] = []
        for _, row in df_ts.iterrows():
            comb_idx = len(player_id_map)
            player_id_map[comb_idx] = str(row.get("player", row.get("sensor_id", row.get("xID", comb_idx))))
            team_players[team_name].append(comb_idx)
    if pos_data:
        framerate = int(pos_data[0].framerate) if pos_data[0].framerate else 25
elif fmt == "dfl":
    pos_data, _, _, teamsheets, _ = dfl.read_position_data_xml(positions_filepath, metadata_filepath) # pos_data is Dict[half_name, Dict[team_name, XY]] and teamsheets is Dict[team_name, Teamsheet]
    for team_name, team_ts in teamsheets.items():
        if team_name == "Ball":
            continue
        df_ts = team_ts.teamsheet.sort_values("xID")
        team_players[team_name] = []
        for _, row in df_ts.iterrows():
            comb_idx = len(player_id_map)
            player_id_map[comb_idx] = str(row.get("player", row.get("pID", comb_idx))) # TODO: check name columns
            team_players[team_name].append(comb_idx)

    first_xy = next(iter(next(iter(pos_data.values())).values()))
    framerate = int(first_xy.framerate) if first_xy.framerate else 25


### KPI Computation

In [ ]:
all_frame_kpis = {}  # {absolute_frame_idx: [[player_id, dist, vel, metpow], ...]}
frame_offset = 0

if fmt == "kinexon":
    # pos_data is List[XY], one entry per group/team (including ball if tracked).
    team_kpi_arrays = {}
    n_frames = None
    for i, xy_obj in enumerate(pos_data):
        team_name = teamsheets[i].teamsheet["tID"].iloc[0] if not teamsheets[i].teamsheet.empty else f"team_{i+1}"
        dist_mod = DistanceModel()
        vel_mod = VelocityModel()
        metpow_mod = MetabolicPowerModel()
        dist_mod.fit(xy_obj)
        vel_mod.fit(xy_obj)
        metpow_mod.fit(xy_obj)
        dist_arr = np.array(dist_mod.cumulative_distance_covered()).round(2)  # (T, N)
        vel_arr = np.array(vel_mod.velocity()).round(2)                       # (T, N)
        metpow_arr = np.array(metpow_mod.metabolic_power()).round(2)          # (T, N)
        team_kpi_arrays[team_name] = (dist_arr, vel_arr, metpow_arr)
        if n_frames is None:
            n_frames = dist_arr.shape[0]
    if n_frames is not None:
        for team_name, (dist_arr, vel_arr, metpow_arr) in team_kpi_arrays.items():
            team_comb_ids = team_players.get(team_name, [])
            n_players = dist_arr.shape[1]
            dist_list = dist_arr.tolist()
            vel_list = vel_arr.tolist()
            metpow_list = metpow_arr.tolist()
            for frame_idx in range(n_frames):
                if frame_idx not in all_frame_kpis:
                    all_frame_kpis[frame_idx] = []
                for p in range(n_players):
                    pid = player_id_map.get(
                        team_comb_ids[p] if p < len(team_comb_ids) else -1,
                        f"{team_name}_p{p}"
                    )
                    d = dist_list[frame_idx][p]
                    v = vel_list[frame_idx][p]
                    m = metpow_list[frame_idx][p]
                    all_frame_kpis[frame_idx].append([
                        pid,
                        None if d != d else d,   # NaN → None (NaN != NaN is always True)
                        None if v != v else v,
                        None if m != m else m,
                    ])
elif fmt == "dfl":
    # pos_data is Dict[half_name → Dict[team_name → XY]].
    # Halves are concatenated into a flat frame index using frame_offset.
    # Ball is included as a regular group; its player_id falls back to "Ball_p0"
    # since it has no teamsheet entry.
    for half_name, teams_dict in pos_data.items():
        team_kpi_arrays = {}
        n_frames = None
        for team_name, xy_obj in teams_dict.items():
            dist_mod = DistanceModel()
            vel_mod = VelocityModel()
            metpow_mod = MetabolicPowerModel()
            dist_mod.fit(xy_obj)
            vel_mod.fit(xy_obj)
            metpow_mod.fit(xy_obj)
            dist_arr = np.array(dist_mod.cumulative_distance_covered()).round(2)  # (T, N)
            vel_arr = np.array(vel_mod.velocity()).round(2)                       # (T, N)
            metpow_arr = np.array(metpow_mod.metabolic_power()).round(2)          # (T, N)
            team_kpi_arrays[team_name] = (dist_arr, vel_arr, metpow_arr)
            if n_frames is None:
                n_frames = dist_arr.shape[0]
        if n_frames is None:
            continue
        for team_name, (dist_arr, vel_arr, metpow_arr) in team_kpi_arrays.items():
            team_comb_ids = team_players.get(team_name, [])
            n_players = dist_arr.shape[1]
            dist_list = dist_arr.tolist()
            vel_list = vel_arr.tolist()
            metpow_list = metpow_arr.tolist()
            for frame_idx in range(n_frames):
                abs_frame = frame_offset + frame_idx
                if abs_frame not in all_frame_kpis:
                    all_frame_kpis[abs_frame] = []
                for p in range(n_players):
                    pid = player_id_map.get(
                        team_comb_ids[p] if p < len(team_comb_ids) else -1,
                        f"{team_name}_p{p}"
                    )
                    d = dist_list[frame_idx][p]
                    v = vel_list[frame_idx][p]
                    m = metpow_list[frame_idx][p]
                    all_frame_kpis[abs_frame].append([
                        pid,
                        None if d != d else d,   # NaN → None (NaN != NaN is always True)
                        None if v != v else v,
                        None if m != m else m,
                    ])
        frame_offset += n_frames


In [153]:
from floodlight.models.space import DiscreteVoronoiModel

ModuleNotFoundError: No module named 'floodlight.models.space'

### Output

In [145]:
kpi_data

{0: [['B Kumarli', 0.0, None, None],
  ['B Wohlan', 0.0, None, None],
  ['B Steinhausen', 0.0, None, None],
  ['B Lasic', 0.0, None, None],
  ['B Reichelt', 0.0, None, None],
  ['B Koch', 0.0, None, None],
  ['B Haeck', 0.0, None, None],
  ['B Reitmeier', 0.0, None, None],
  ['B Vosen', 0.0, None, None],
  ['B Dolenga', 0.0, None, None],
  ['B Wittkugel', 0.0, None, None],
  ['B Vogel', 0.0, None, None],
  ['B Adam', 0.0, None, None],
  ['B Beckschulte', 0.0, None, None],
  ['B Jendrossek', 0.0, None, None],
  ['B Barut', 0.0, None, None],
  ['B Stoermann', 0.0, None, None],
  ['B Stegemann', 0.0, None, None],
  ['B Ortner', 0.0, None, None],
  ['B Neugebauer', 0.0, None, None],
  ['B Buecken', 0.0, None, None],
  ['B Welter', 0.0, None, None],
  [' Ball 1', 0.0, 0.09, 0.76]],
 1: [['B Kumarli', 0.0, None, None],
  ['B Wohlan', 0.0, None, None],
  ['B Steinhausen', 0.0, None, None],
  ['B Lasic', 0.0, None, None],
  ['B Reichelt', 0.0, None, None],
  ['B Koch', 0.0, None, None],
  ['B 

In [146]:
meta = {
    "format": fmt,
    "kpi_names": ["distance_covered", "velocity", "metabolic_power"],
    "player_ids": {str(k): v for k, v in player_id_map.items()},
    "teams": {k: [int(x) for x in v] for k, v in team_players.items()},
    "segments": segment_meta,
    "framerate": framerate,
    "tracking_data_id": parameters.get("tracking_data_id"),
}
